# T6: 2026 R02 中国GP — FP予選シミュレーション vs 実際の予選結果

## 概要
R02中国GPはスプリントウィークエンドのためFP1のみ。  
FP1の予選シミュレーションラップ（CLAUDE.md準拠の個別ラップベース識別）から  
各ドライバーのFP予測ラップタイムを算出し、実際の予選結果と比較する。

### 予選シミュレーション識別ロジック（CLAUDE.md準拠）
1. Softタイヤで記録されたラップ
2. アウトラップ（PitOutTime_secが存在）を除外
3. TyreLife <= 8（新品〜浅い使用状態）
4. セッション最速の103%以内（本気アタックラップ）

## セットアップ & データ読み込み

In [ ]:
import matplotlib
matplotlib.use('Agg')  # GUIなし環境対応

import csv
import os
import math
import statistics
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

# ===== パス設定 =====
BASE_DIR = '/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised'
FP_CSV = os.path.join(BASE_DIR, 'data/2026_R02_China/export/fp_laps.csv')
QUALI_CSV = os.path.join(BASE_DIR, 'data/2026_R02_China/export/quali_laps.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'notebooks/output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== グラフスタイル =====
STYLE = {
    'bg_color': '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize': (12, 6.75),
    'title_size': 18,
    'label_size': 12,
}

# ===== CSV読み込みヘルパー =====
def load_csv(path):
    """CSVをdict listとして読み込む"""
    with open(path, encoding='utf-8') as f:
        return list(csv.DictReader(f))

def to_float(val, default=None):
    """文字列→float変換（空文字・NaN対応）"""
    try:
        v = float(val)
        return v if not math.isnan(v) else default
    except (TypeError, ValueError):
        return default

# FPデータ読み込み
fp_rows = load_csv(FP_CSV)
quali_rows = load_csv(QUALI_CSV)

# セッション種別を動的に確認
sessions_found = sorted(set(r['Session'] for r in fp_rows))
print(f"総FPラップ数: {len(fp_rows)}")
print(f"検出されたセッション: {sessions_found}")
print(f"予選総ラップ数: {len(quali_rows)}")

## 予選シミュレーションラップ特定（CLAUDE.md準拠ロジック）

In [ ]:
def identify_qualisim_laps(rows, session_name=None):
    """
    予選シミュレーションラップを特定する
    1. Softタイヤのみ
    2. アウトラップ除外（PitOutTime_sec存在）
    3. TyreLife <= 8
    4. セッション最速の103%以内
    """
    # セッションフィルタ
    target = [r for r in rows if session_name is None or r['Session'] == session_name]

    # ステップ1: Softタイヤのみ（LapTimeが存在するもの）
    soft_laps = [
        r for r in target
        if r['Compound'] == 'SOFT' and to_float(r['LapTime_sec']) is not None
    ]
    print(f"  [{session_name}] Soft全ラップ: {len(soft_laps)}")

    # ステップ2: アウトラップ除外（PitOutTime_secが空でないもの = アウトラップ）
    no_outlap = [
        r for r in soft_laps
        if not r.get('PitOutTime_sec', '').strip()
    ]
    print(f"  [{session_name}] アウトラップ除外後: {len(no_outlap)}")

    # ステップ3: TyreLife <= 8
    fresh_laps = [
        r for r in no_outlap
        if to_float(r['TyreLife'], 999) <= 8
    ]
    print(f"  [{session_name}] TyreLife<=8後: {len(fresh_laps)}")

    # ステップ4: セッション最速の103%以内
    valid_times = [to_float(r['LapTime_sec']) for r in fresh_laps]
    valid_times = [t for t in valid_times if t is not None]
    if not valid_times:
        return []
    session_best = min(valid_times)
    threshold = session_best * 1.03
    print(f"  [{session_name}] セッションベスト: {session_best:.3f}秒, 103%閾値: {threshold:.3f}秒")

    qualisim = [
        r for r in fresh_laps
        if to_float(r['LapTime_sec'], 9999) <= threshold
    ]
    print(f"  [{session_name}] 最終: {len(qualisim)}ラップ特定")
    return qualisim

# セッション別に予選シミュラップを特定
qualisim_by_session = {}
for sess in sessions_found:
    print(f"\nセッション: {sess}")
    laps = identify_qualisim_laps(fp_rows, sess)
    qualisim_by_session[sess] = laps

# 全セッション統合
all_qualisim = []
for laps in qualisim_by_session.values():
    all_qualisim.extend(laps)
print(f"\n全セッション合計: {len(all_qualisim)}ラップ")

## ドライバー別FP予測ラップタイム